In [1]:
# 1. Define filter dictionary and extract it the results via duckdb queries.
import os
import numpy as np
import pandas as pd
import duckdb as db
from dotenv import load_dotenv

load_dotenv()

# Initialize DuckDB connection
con = db.connect()
con.execute("INSTALL postgres; LOAD postgres;")

attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={os.getenv("PGDATABASE")} 
        user={os.getenv("PGUSER")} 
        password={os.getenv("PGPASSWORD")} 
        host={os.getenv("PGHOST")} 
        port={os.getenv("PGPORT")}' 
AS db (TYPE postgres);
"""
con.execute(attach_query)

# A. Register updated DuckDB filters (preserving code 6 as a structural valid category)
con.execute("""
    -- 1-digit macro: preserves code 6 (Not applicable), maps 7, 8, 9 to NULL
    CREATE OR REPLACE MACRO clean_ess_1d(x) AS 
        CASE 
            WHEN x IN (7, 8, 9) THEN NULL 
            ELSE x 
        END;
        
    -- 4-digit macro for education levels
    CREATE OR REPLACE MACRO clean_ess_4d(x) AS 
        CASE WHEN x IN (7777, 8888, 9999) THEN NULL ELSE x END;
        
    -- 5-digit macro for employee counts
    CREATE OR REPLACE MACRO clean_ess_5d(x) AS 
        CASE WHEN x IN (66666, 77777, 88888, 99999) THEN NULL ELSE x END;
""")

# B. Build and execute the query for our variables
query_custom_3vars = """
CREATE OR REPLACE TABLE surveys_sample_3v AS
SELECT
    s.idno,
    s.cntry,
    s.essround,
    s.emplrel AS emplrel_raw, -- Keep raw emplrel for tracking routing vs missing
    s.emplno AS emplno_raw, -- keep raw emplno for tracking routing vs missing
    clean_ess_1d(s.emplrel) AS emplrel,  -- Cleaned 1D (code 6 is preserved, 7-9 mapped to NULL)
    clean_ess_4d(s.edulvlb) AS edulvlb,  -- Cleaned 4D
    clean_ess_5d(s.emplno)  AS emplno    -- Cleaned 5D
FROM db.surveys s
WHERE s.cntry IS NOT NULL;
"""
con.execute(query_custom_3vars)

# C. Extract results to Pandas DataFrame
df_sample_3v = con.sql("SELECT * FROM surveys_sample_3v").df()

# D. Feature Engineering: Create 'EmployType' 
# Value 1: Employee (Wage earner)
# Value 2: Self-employed or family business (Self Employers / Businessmen)
# Value 3: Inactive, unemployed, or non-applicable population (Default)
df_sample_3v['EmployType'] = 3  # Set default value first
df_sample_3v.loc[df_sample_3v['emplrel'] == 1, 'EmployType'] = 1
df_sample_3v.loc[df_sample_3v['emplrel'].isin([2, 3]), 'EmployType'] = 2

# E. Structural assignment for emplno: 
# If the respondent is a wage employee (EmployType == 1), 
# their number of employees is structurally 0, not a missing value to impute.
df_sample_3v.loc[df_sample_3v['EmployType'] == 1, 'emplno'] = df_sample_3v['emplno'].fillna(0)

print(f"✅ ¡Sample dataframe successfully processed with proper routing separation!")
print(f"📊 Final dimensions: {df_sample_3v.shape}")
print("\nDistribution of 'EmployType' (1: Employee, 2: Self-employed, 3: Inactive/Not applicable):")
print(df_sample_3v['EmployType'].value_counts(sort=False))

print("\nFirst lines with clean results and new feature:")
print(df_sample_3v[['cntry', 'emplrel', 'EmployType', 'emplno', 'edulvlb']].head(10))

✅ ¡Sample dataframe successfully processed with proper routing separation!
📊 Final dimensions: (41702, 9)

Distribution of 'EmployType' (1: Employee, 2: Self-employed, 3: Inactive/Not applicable):
EmployType
3     4186
1    32137
2     5379
Name: count, dtype: int64

First lines with clean results and new feature:
  cntry  emplrel  EmployType  emplno  edulvlb
0    AT        6           3    <NA>      322
1    AT        1           1       0      423
2    AT        1           1       0      610
3    AT        1           1       0      422
4    AT        1           1       0      322
5    AT        1           1       0      313
6    AT        1           1       0      212
7    AT        1           1       0      720
8    AT        1           1       0      422
9    AT        1           1       0      212


In [2]:
df_sample_3v.head(10)


,idno,cntry,essround,emplrel_raw,emplno_raw,emplrel,edulvlb,emplno,EmployType
0,50014,AT,11,6,66666,6,322,<NA>,3
1,50030,AT,11,1,66666,1,423,0,1
2,50057,AT,11,1,66666,1,610,0,1
3,50106,AT,11,1,66666,1,422,0,1
4,50145,AT,11,1,66666,1,322,0,1
5,50158,AT,11,1,66666,1,313,0,1
6,50211,AT,11,1,66666,1,212,0,1
7,50212,AT,11,1,66666,1,720,0,1
8,50213,AT,11,1,66666,1,422,0,1
9,50235,AT,11,1,66666,1,212,0,1


In [3]:
# 1. Añadir columnas si no existen
#con.execute("ALTER TABLE db.surveys ADD COLUMN IF NOT EXISTS emplrel_raw INT;")
#con.execute("ALTER TABLE db.surveys ADD COLUMN IF NOT EXISTS EmployType INT;")
con.execute("ALTER TABLE db.surveys ADD COLUMN IF NOT EXISTS emplno_raw INT;")


In [5]:
# 2. Ahora sí, el UPDATE
con.execute("""
    UPDATE db.surveys s
    SET
        emplrel_raw = u.emplrel_raw,
        emplrel     = u.emplrel,
        emplno_raw  = u.emplno_raw,   -- si la tienes en surveys_sample_3v
        emplno      = u.emplno,
        EmployType  = u.EmployType
    FROM df_sample_3v u
    WHERE s.idno = u.idno
""")  
print(con.sql("""
    SELECT COUNT(*) AS total,
           COUNT(emplrel_raw) AS has_raw,
           COUNT(EmployType)  AS has_type
    FROM db.surveys
    WHERE emplrel_raw IS NOT NULL
""").df())

   total  has_raw  has_type
0  41702    41702     41702


In [6]:
del df_sample_3v,